In [ ]:
import json, sys
from collections import Counter, defaultdict
from dataclasses import asdict
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import recovery as R

# Reruns notebook 10's S3 Runs under wrong_record a second time, after a third apir-adapter fix. Notebook 12's
# rerun cured S3's slower queries under abort, but not under switch_replica: a repetition there builds up to four
# shadow queries of 1.9 GB each, collected only at the next repetition, and its timed further attempts ran
# alongside that garbage (notebook 11, cell 7: query generation 0.78 to 0.89 s against 0.59 s, a further attempt
# 2.5 s against 2.0 s). The adapter now collects it right after each injected fault, untimed. Notebook 11 takes
# these Runs from here, M1's switch_replica Runs from notebook 12, and every other Run from notebook 10. The other
# arms' shadow queries are small, and their points already match the computed cost within 1 % (notebook 11, cell 5).
DAY = repo / "results/raw/rq4-day-20261111"
day = [R.Run(**json.loads(l)) for l in (DAY / "plan.jsonl").read_text().splitlines()]

def changed(r):
    return r.arm == "S3" and r.fault == "wrong_record" and r.fault_prob > 0

rerun = [r for r in day if changed(r)]
day_wall = {json.loads(l)["name"]: json.loads(l)["wall_s"] for l in (DAY / "manifest.jsonl").read_text().splitlines()}
print(f"{len(rerun)} of {len(day)} Runs rerun; they took {sum(day_wall[r.name] for r in rerun) / 60:.0f} min in notebook 10")
for r in rerun:
    print(f"  {r.name}  ({day_wall[r.name]:.0f} s)")

In [ ]:
# About 40 minutes of adapter runs, serially, each under caffeinate, the 14 GiB cap and a hygiene watch.
# Rerunning this cell resumes the same run id.
cfg = R.load_config(R.DEFAULT_CONFIG)
RUN_ID = "rq4-rerun-s3-20261111"
ex = R.prepare(rerun, cfg, R.DEFAULT_CONFIG, run_id=RUN_ID)
plan_file = ex.out_dir / "plan.jsonl"
if not plan_file.exists():
    plan_file.write_text("".join(json.dumps(asdict(r)) + "\n" for r in rerun))
assert [json.loads(l) for l in plan_file.read_text().splitlines()] == [asdict(r) for r in rerun], "the plan changed"
status = R.run_and_report(rerun, ex)

In [ ]:
rows = [json.loads(l) for l in (ex.out_dir / "manifest.jsonl").read_text().splitlines()]
last = {r["name"]: r for r in rows}                     # a resumed Run's latest attempt
outcomes = Counter(r["outcome"] for r in last.values())
print(f"{len(last)} of {len(rerun)} Runs attempted: {dict(outcomes)}; citable {sum(r['citable'] for r in last.values())}")
for r in last.values():
    print(f"  {r['outcome']:3s} {r['name']}  {r['wall_s']:.0f} s (notebook 10: {day_wall[r['name']]:.0f} s)  "
          f"peak {r['peak_footprint_bytes'] / 2**30:.2f} GiB")
print(f"adapter wall time {sum(r['wall_s'] for r in last.values()) / 3600:.2f} h")